In [1]:
import prun_data_frames as prundf
from prun_data_frames import CX, Currency
from price import EvaluatedPrices, PriceOverride, PriceType, OverrideType
from labor import EvaluatedLabor
from analysis import BuildingAnalysis
import polars as pl
import polars.selectors as cs

In [2]:
prices = prundf.PrunPrices()

# enter price overrides here
# overrides = [PriceOverride("AR",CX.CI1,20,OverrideType.BUY)]

overrides = []

cx = CX.CI1

evaluated_prices = EvaluatedPrices(prices, cx, overrides=overrides)

labor = EvaluatedLabor(evaluated_prices)

buildings = prundf.PrunBuildings()

analysis = BuildingAnalysis(cx=cx, buildings=prundf.PrunBuildings(), prices=evaluated_prices, materials=prundf.PrunMaterials(), labor=labor)

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(-1)

polars.config.Config

In [17]:
a_df = analysis.df(1.40)
a_df = (a_df.join(buildings.source_df, left_on=("Building"), right_on="Ticker")
        .drop("Area","Name"))
prices_df = prices.source_df.select(["Ticker",cs.starts_with(cx.name)])
print(prices_df)
a_df = (a_df.join(evaluated_prices.df,left_on="OutputMaterial", right_on="Ticker"))
        
#print(analysis.unit_efficiency_df.filter(pl.col("Recipe").str.contains("SF").and_(pl.col("Building") == "REF")))
print(a_df
        .filter(pl.col("Expertise")=="MANUFACTURING")
        .filter(pl.col("Building")!="SPF")
        #.filter(pl.col("TotalCostPerDay")<30000)
        .sort("ProfitPerDay", descending=True)
        .head(10))

shape: (369, 8)
┌────────┬──────────────┬────────────┬──────────────┬──────────────┬────────────┬──────────────┬──────────────┐
│ Ticker ┆ CI1-Average  ┆ CI1-AskAmt ┆ CI1-AskPrice ┆ CI1-AskAvail ┆ CI1-BidAmt ┆ CI1-BidPrice ┆ CI1-BidAvail │
│ ---    ┆ ---          ┆ ---        ┆ ---          ┆ ---          ┆ ---        ┆ ---          ┆ ---          │
│ str    ┆ f64          ┆ i64        ┆ f64          ┆ i64          ┆ i64        ┆ f64          ┆ i64          │
╞════════╪══════════════╪════════════╪══════════════╪══════════════╪════════════╪══════════════╪══════════════╡
│ AAR    ┆ 16200.0      ┆ 96         ┆ 16200.0      ┆ 277          ┆ 70         ┆ 14000.0      ┆ 70           │
│ ABH    ┆ 53900.0      ┆ 42         ┆ 53900.0      ┆ 888          ┆ 67         ┆ 52000.0      ┆ 260          │
│ ACS    ┆ 157124.71875 ┆ 9          ┆ 170000.0     ┆ 9            ┆ 10         ┆ 150000.0     ┆ 40           │
│ ADE    ┆ 43800.0      ┆ 18         ┆ 43000.0      ┆ 1300         ┆ 100        ┆ 40100.

In [18]:
print(analysis.df(efficiency=1.25).filter(pl.col("Recipe").str.contains("SF").and_(pl.col("Building") == "REF")))

shape: (2, 15)
┌──────────┬──────────┬─────────────────┬─────────────────────────┬────────────┬─────────────┬────────────────┬───────────────┬────────────────┬───────────────┬─────────────────┬──────────────────────┬─────────────────┬─────────────┬──────────────┐
│ Building ┆ Duration ┆ InputCostPerRun ┆ Recipe                  ┆ RunsPerDay ┆ UnitsPerDay ┆ OutputMaterial ┆ RevenuePerRun ┆ OutputQuantity ┆ RevenuePerDay ┆ InputCostPerDay ┆ TotalLaborCostPerDay ┆ TotalCostPerDay ┆ CostPerUnit ┆ ProfitPerDay │
│ ---      ┆ ---      ┆ ---             ┆ ---                     ┆ ---        ┆ ---         ┆ ---            ┆ ---           ┆ ---            ┆ ---           ┆ ---             ┆ ---                  ┆ ---             ┆ ---         ┆ ---          │
│ str      ┆ f64      ┆ f64             ┆ str                     ┆ f64        ┆ f64         ┆ str            ┆ f64           ┆ i64            ┆ f64           ┆ f64             ┆ f64                  ┆ f64             ┆ f64         ┆ f64 